In [1]:
%load_ext autoreload
%autoreload 2

In [68]:
from concept_abstraction.training import train_ppo_model, SimpleQEstimator
from concept_abstraction.selection import greedy_selection_supervised
from concept_abstraction.env_utils import *
from concept_abstraction.utils import *
import sys 
import argparse
import secrets
import numpy as np 
import random 
import time 
from collections import Counter
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score


In [3]:
is_jupyter = 'ipykernel' in sys.modules

In [93]:
if is_jupyter: 
    seed        = 42
    environment_string = "cart_pole_binary"
    training_timesteps = 10000
    num_concepts_selected = 20
    selection_function = "policy"
    human_accuracy_by_concept = None 
    cbm_accuracy_by_concept = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1] 
    human_reliance_by_concept = None  
    target_abstraction = 0.05
    out_folder = "cart_pole"
    reward_error = 0
else:
    parser = argparse.ArgumentParser()
    parser.add_argument('--seed', help='Random Seed', type=int, default=42)
    parser.add_argument('--environment_string', help='Which environment to create', type=str, default="tree")
    parser.add_argument('--training_timesteps', help='Number of training timesteps', type=int, default=10000)
    parser.add_argument('--num_concepts_selected', help='Number of concepts selected by greedy or random',type=int, default=0)
    parser.add_argument('--selection_function', help='When selecting, use q_value, policy, or transition?', type=str, default="policy")
    parser.add_argument('--target_abstraction', help='Value for the target abstraction with human performance', type=float, default=0.05)
    parser.add_argument('--human_accuracy_by_concept', nargs='*', type=float, default=None)
    parser.add_argument('--cbm_accuracy_by_concept', help="What is the accuracy of AI per concept?", nargs='*', type=float, default=None)
    parser.add_argument('--human_reliance_by_concept', help="How much does AI rely on human intervention?",  nargs='*', type=float, default=None)
    parser.add_argument('--reward_error', help="How much to perturb the reward by?", type=float, default=0)
    parser.add_argument('--out_folder', help='Which folder', type=str, default="exploration")

    args = parser.parse_args()

    seed = args.seed
    environment_string = args.environment_string
    training_timesteps = args.training_timesteps 
    selection_function = args.selection_function
    num_concepts_selected = args.num_concepts_selected
    human_accuracy_by_concept = args.human_accuracy_by_concept
    human_reliance_by_concept = args.human_reliance_by_concept
    target_abstraction = args.target_abstraction
    cbm_accuracy_by_concept = args.cbm_accuracy_by_concept
    reward_error = args.reward_error
    out_folder = args.out_folder

save_name = secrets.token_hex(4)  

In [5]:
results = {}
results['parameters'] = {'seed'      : seed,
        'environment_string'    : environment_string, 
        'training_timesteps': training_timesteps, 
        'selection_function': selection_function,
        'num_concepts_selected': num_concepts_selected,
        'human_accuracy_by_concept': human_accuracy_by_concept, 
        'human_reliance_by_concept': human_reliance_by_concept, 
        'target_abstraction': target_abstraction,
        'cbm_accuracy_by_concept': cbm_accuracy_by_concept,
        'reward_error': reward_error, 
}
print("Parameters {}".format(results['parameters']))

Parameters {'seed': 42, 'environment_string': 'cart_pole_binary', 'training_timesteps': 10000, 'selection_function': 'policy', 'num_concepts_selected': 0, 'human_accuracy_by_concept': None, 'human_reliance_by_concept': None, 'target_abstraction': 0.05, 'cbm_accuracy_by_concept': [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1], 'reward_error': 0}


In [6]:
np.random.seed(seed)
random.seed(seed)

In [7]:
dataset = json.load(open("../../data/cub/preprocessed.json"))

## Concept Selection

In [45]:
train_X = np.array([row['attributes'] for row in dataset['train']])
test_X = np.array([row['attributes'] for row in dataset['test']])

In [53]:
all_rows_train = set([''.join([str(int(j)) for j in row]) for row in train_X])
all_rows_test = set([''.join([str(int(j)) for j in row]) for row in test_X])

In [59]:
def get_performance(selected_concepts,accuracy_by_concept):
    train_X = np.array([row['attributes'] for row in dataset['train']])
    test_X = np.array([row['attributes'] for row in dataset['test']])

    flip_probs = 1 - np.array(accuracy_by_concept)
    rand_vals = np.random.rand(*train_X.shape)
    flip_mask = rand_vals < flip_probs  # True means flip
    train_X = np.where(flip_mask, 1 - train_X, train_X)
    train_X = train_X[:,selected_concepts]

    flip_probs = 1 - np.array(accuracy_by_concept)
    rand_vals = np.random.rand(*test_X.shape)
    flip_mask = rand_vals < flip_probs  # True means flip
    test_X = np.where(flip_mask, 1 - test_X, test_X)
    test_X = test_X[:,selected_concepts]


    train_Y = np.array([row['label'] for row in dataset['train']])
    test_Y = np.array([row['label'] for row in dataset['test']])

    mlp = MLPClassifier(
        hidden_layer_sizes=(50),
        activation='relu',
        solver='adam',
        max_iter=1000,  # increase if needed
        random_state=0,
        alpha=1e-3,  # instead of 0.0001,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=20
    )

    # Train the model
    mlp.fit(train_X, train_Y)

    # Predict on the test set
    y_pred = mlp.predict(test_X)

    # Compute accuracy
    acc = accuracy_score(test_Y, y_pred)
    return acc

In [64]:
train_X = np.array([row['attributes'] for row in dataset['train']])
test_X = np.array([row['attributes'] for row in dataset['test']])
train_Y = np.array([row['label'] for row in dataset['train']])
test_Y = np.array([row['label'] for row in dataset['test']])


In [99]:
selected_concepts = []
random_average_rewards = []
start = time.time() 
for k in range(1,num_concepts_selected+1):
    if k > train_X.shape[1]:
        break 
    
    random_concepts = np.random.choice(list(range(train_X.shape[1])),k,replace=False)
    selected_concepts.append(random_concepts)
    random_average_rewards.append(get_performance(random_concepts,np.ones(312)))

results['random_selection'] = {
    'concepts': selected_concepts, 
    'values': random_average_rewards,
}

In [100]:
results['random_selection']

{'concepts': [array([184]),
  array([30, 24]),
  array([  3, 113, 290]),
  array([219, 287, 129, 193]),
  array([ 75, 257, 129, 137, 205]),
  array([147, 163, 292,   5,  42, 309]),
  array([305,  91, 261,  79, 190,   8, 234]),
  array([269,  40, 128, 260, 221, 184, 144, 172]),
  array([102,  28,  80, 228,  32,  21, 138, 222,  90]),
  array([  3,  38,  27, 135,  89, 308,  37, 196, 280,  77]),
  array([124, 171,  43, 266, 262, 247, 223, 125, 263, 298, 289]),
  array([132,  84, 177,  63,  31, 155,  66, 190, 246, 153,  22, 258]),
  array([116, 101, 123,  14, 207, 170, 110, 273,  76, 308, 108,  97, 214]),
  array([110, 203,  60, 159, 202,  27, 297, 144, 262, 296, 265, 217, 124,
         259])],
 'values': [0.005005177770107007,
  0.012599240593717639,
  0.012944425267518123,
  0.012426648256817397,
  0.00742147048671039,
  0.03175698998964446,
  0.026406627545736968,
  0.026924404556437694,
  0.03279254401104591,
  0.029513289609941318,
  0.03969623748705557,
  0.04159475319295823,
  0.0593

In [94]:
selected_concepts = []
greedy_average_rewards = []
greedy_times = []
start = time.time() 
for k in range(1,num_concepts_selected+1):
    if k > train_X.shape[1]:
        break 
    
    greedy_concepts = greedy_selection_supervised(train_X,train_Y,k)
    selected_concepts.append(greedy_concepts)
    greedy_average_rewards.append(get_performance(greedy_concepts,np.ones(312)))

results['greedy_selection'] = {
    'concepts': selected_concepts, 
    'values': greedy_average_rewards,
}

KeyboardInterrupt: 

In [95]:
results['greedy_selection'] = {
    'concepts': selected_concepts, 
    'values': greedy_average_rewards,
}

In [96]:
results['greedy_selection']

{'concepts': [[20],
  [20, 21],
  [20, 21, 163],
  [20, 21, 163, 7],
  [20, 21, 163, 7, 254],
  [20, 21, 163, 7, 254, 25],
  [20, 21, 163, 7, 254, 25, 9],
  [20, 21, 163, 7, 254, 25, 9, 0],
  [20, 21, 163, 7, 254, 25, 9, 0, 1],
  [20, 21, 163, 7, 254, 25, 9, 0, 1, 14],
  [20, 21, 163, 7, 254, 25, 9, 0, 1, 14, 2],
  [20, 21, 163, 7, 254, 25, 9, 0, 1, 14, 2, 3],
  [20, 21, 163, 7, 254, 25, 9, 0, 1, 14, 2, 3, 53],
  [20, 21, 163, 7, 254, 25, 9, 0, 1, 14, 2, 3, 53, 4]],
 'values': [0.005005177770107007,
  0.018812564722126338,
  0.03434587504314809,
  0.04263030721435968,
  0.05402140144977563,
  0.06679323438039352,
  0.07611322057300655,
  0.07801173627890921,
  0.08767690714532275,
  0.09820503969623749,
  0.09406282361063169,
  0.10545391784604763,
  0.10579910251984811,
  0.11477390403866068]}

In [ ]:
all_attributes = open("../../data/cub/attributes.txt").read().split("\n")

In [ ]:
[all_attributes[i] for i in results['greedy_selection']['concepts'][-1]]

['21 has_wing_color::black',
 '22 has_wing_color::white',
 '164 has_forehead_color::black',
 '8 has_bill_shape::cone',
 '255 has_primary_color::yellow',
 '26 has_upperparts_color::brown',
 '10 has_wing_color::blue',
 '1 has_bill_shape::curved_(up_or_down)',
 '2 has_bill_shape::dagger',
 '15 has_wing_color::grey']

## Performance under Uncertainty

In [15]:
if human_accuracy_by_concept is not None or cbm_accuracy_by_concept is not None:
    if human_accuracy_by_concept is None:
        modified_acc_rate = cbm_accuracy_by_concept
    elif cbm_accuracy_by_concept is None:
        modified_acc_rate = human_accuracy_by_concept
    else:
        modified_acc_rate = [reliance_percent*human_acc + (1-reliance_percent)*machine_acc 
                for human_acc,machine_acc,reliance_percent in zip(human_accuracy_by_concept,
                                                                cbm_accuracy_by_concept,
                                                                human_reliance_by_concept)]

    selected_concepts = human_centered_selection_real_world(env,np.array(modified_acc_rate),target_abstraction,np.array(all_binarized_concepts),state_values)
    selected_concepts = [idx for idx,i in enumerate(selected_concepts) if i>=0.5]


    env = create_environment_from_string_real_world(environment_string,selected_concepts,accuracies=modified_acc_rate)
    model = train_ppo_model(env,total_timesteps=training_timesteps)
    env = create_environment_from_string_real_world(environment_string,selected_concepts,accuracies=None)
    human_perf = get_average_reward(env,model)

    results['uncertainty'] = {
        'selected_concepts': selected_concepts,
        'combined_accuracies': modified_acc_rate,
        'combined_value': human_perf,
    }

Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-14
Set parameter DualReductions to value 0
Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (linux64)

CPU model: Intel(R) Core(TM) i7-7700K CPU @ 4.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 38 rows, 17 columns and 139 nonzeros
Model fingerprint: 0xd0f5ed7b
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e-01, 1e-01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve time: 0.00s
Presolved: 24 rows, 17 columns, 79 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.0000000e-01   2.906000e+01   0.000000e+00      0s
      16    1.0000000e-01   0.000000e+00   0.000000e+00      0s

Solved in 16 iterations and 0.00 seconds (0.00 work units)
Optimal objective  1.000000000e-01


## Save Data

In [16]:
save_path = get_save_path(out_folder,save_name)

In [17]:
delete_duplicate_results(out_folder,"",results)

In [18]:
json.dump(results,open('../../results/'+save_path,'w'))